In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
import kagglehub

# Downloading the 2015-2018 directly from kaggle
for i in range(2):
  path_old = kagglehub.dataset_download("ankkur13/boston-crime-data")
  if i == 1:
    print(path_old)

# Downloading the 2015-2018 directly from kaggle
for i in range(2):
  path_new = kagglehub.dataset_download("adillaifpe/boston-crime-dataset-2023-2025")
  if i == 1:
    print(path_new)

def get_data():
  data = pd.read_csv('/kaggle/input/boston-crime-data/crime.csv', encoding='latin-1')
  return data

# upload the 2023-2025 dataset
def new_get_data():
  data = pd.read_csv('/kaggle/input//boston-crime-dataset-2023-2025/Boston_Incidents_2023-2025_filtered.csv', encoding='latin-1')
  return data

100%|██████████| 10.7M/10.7M [00:01<00:00, 6.22MB/s]

Extracting files...


Using Colab cache for faster access to the 'boston-crime-data' dataset.
/kaggle/input/boston-crime-data


100%|██████████| 10.3M/10.3M [00:02<00:00, 4.13MB/s]

Extracting files...


Using Colab cache for faster access to the 'boston-crime-dataset-2023-2025' dataset.
/kaggle/input/boston-crime-dataset-2023-2025


In [2]:
# auxiliar function to print all metrics with a prediction and a y_test

def print_metrics(y_test, y_pred, title):
  print('\n\n\n')
  print("# Results for", title)
  print(f"\n       Accuracy: {(accuracy_score(y_test, y_pred)*100):.2f}%")
  print(f"       F1 macro: {(f1_score(y_test, y_pred, average='macro')*100):.2f}%")
  print(f"Precision macro: {(precision_score(y_test, y_pred, average='macro')*100):.2f}%")
  print(f"   Recall macro: {(recall_score(y_test, y_pred, average='macro')*100):.2f}%\n\n\n")

  cm = confusion_matrix(y_test, y_pred)

  parts = []

  for i in range(len(cm)):
      total = 0
      for m in cm[i]:
        total+=m
      parts.append(float((1.0-(cm[i][i]/total)) * 100))

  if len(cm) == 3:
    print(f'  Part One Error: {(parts[0]):.2f}%')
    print(f'  Part Two Error: {(parts[2]):.2f}%')
    print(f'Part Three Error: {(parts[1]):.2f}%')

    print('\n Confusion Matrix')
    print('            Part One     Part Two     Part Three')
    print(f'  Part One      {cm[0][0]}         {cm[0][2]}           {cm[0][1]}')
    print(f'  Part Two      {cm[2][0]}         {cm[2][2]}           {cm[2][1]}')
    print(f'Part Three      {cm[1][0]}         {cm[1][2]}           {cm[1][1]}')
  elif len(cm) == 2:
    print(f'Part One Error: {(parts[0]):.2f}%')
    print(f'Part Two Error: {(parts[1]):.2f}%')
    print('\n Confusion Matrix')
    print('            Part One     Part Two')
    print(f'Part One      {cm[0][0]}         {cm[0][1]}')
    print(f'Part Two      {cm[1][0]}         {cm[1][1]}')

  return

In [3]:
# preprocessing the datasets

def day_to_num(df):
    df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Monday', 2)
    df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Tuesday', 3)
    df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Wednesday', 4)
    df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Thursday', 5)
    df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Friday', 6)
    df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Saturday', 7)
    df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Sunday', 1)

    return df

def district_to_num(df):
  conditions = [
        df['DISTRICT'] == 'A1',
        df['DISTRICT'] == 'A7',
        df['DISTRICT'] == 'A15',
        df['DISTRICT'] == 'B2',
        df['DISTRICT'] == 'B3',
        df['DISTRICT'] == 'C6',
        df['DISTRICT'] == 'C11',
        df['DISTRICT'] == 'D4',
        df['DISTRICT'] == 'D14',
        df['DISTRICT'] == 'E5',
        df['DISTRICT'] == 'E13',
        df['DISTRICT'] == 'E18',
    ]
  choices = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

  df['DISTRICT'] = np.select(conditions, choices, default=0)

  return df

def create_engineered_features(df):
  df['season_hour'] = df['season'] * df['HOUR']

  df['day_district'] = df['DISTRICT'] * df['DAY_OF_WEEK']

  return df

def preprocess_dataset1(df, all_classes=True):
  df = df[['UCR_PART', 'MONTH', 'DAY_OF_WEEK', 'HOUR', 'DISTRICT']]

  df = df.loc[~(df['UCR_PART'] == 'Other')]

  if not all_classes:
    df = df.loc[~(df['UCR_PART'] == 'Part Three')]

  df = df.dropna()

  df = day_to_num(df)
  df = district_to_num(df)

  df['season'] = df['MONTH'] % 12 // 3
  df = create_engineered_features(df)

  return df

def preprocess_dataset2(df, all_classes=True):
  df['UCR_PART'] = df['Crime Part']
  df['DISTRICT'] = df['BPD District']
  df['season'] = df['Quarter']
  df['MONTH'] = df['Month']
  df['DAY_OF_WEEK'] = df['Day of Week']
  df['HOUR'] = df['Hour of Day']

  df = df.loc[~(df['DISTRICT'] == 'External')]
  df = df.loc[~(df['DISTRICT'] == 'Outside of Boston')]

  df = df.loc[~(df['UCR_PART'] == 'Other')]

  if not all_classes:
    df = df.loc[~(df['UCR_PART'] == 'Part Three')]

  df = df[['UCR_PART', 'MONTH', 'DAY_OF_WEEK', 'HOUR', 'DISTRICT', 'season']]

  df = df.dropna()

  df = district_to_num(df)

  df = create_engineered_features(df)

  return df

def undersample(df):

  part_1 = df[df['UCR_PART'] == 'Part One']
  part_2 = df[df['UCR_PART'] == 'Part Two']
  part_3 = df[df['UCR_PART'] == 'Part Three']

  desired_size = min(len(part_1), len(part_2))

  # Undersample Part 3 to the desired size
  part_3_undersampled = resample(part_3,
                                  replace=False,    # Without replacement
                                  n_samples=desired_size,  # New size to match Part 1 or Part 2
                                  random_state=42)  # Set random_state for reproducibility

  part_2_undersampled = resample(part_2,
                                  replace=False,    # Without replacement
                                  n_samples=desired_size,  # New size to match Part 1 or Part 2
                                  random_state=42)  # Set random_state for reproducibility

  # Combine the undersampled Part 3 with Part 1 and Part 2
  df_undersampled = pd.concat([part_1, part_2_undersampled, part_3_undersampled])

  df_undersampled = df_undersampled.dropna()

  return df_undersampled

def oversample(df):
  X = df.drop(columns=['UCR_PART'])
  y = df['UCR_PART']

  X_resampled, y_resampled = SMOTE(sampling_strategy='auto', random_state=42).fit_resample(X, y)

  return X_resampled, y_resampled


def scale_data(df, option=''):

  if option == 'undersample':
    df = undersample(df)
  elif option == 'oversample':
    X, y = oversample(df)

  if option != 'oversample':
    X = df.drop(columns=['UCR_PART'])
    y = df['UCR_PART']

  print(y.value_counts())
  scaler = StandardScaler()
  X = scaler.fit_transform(X)

  return X, y

def separate_confusions(y_test, y_pred):
  parts =[]
  cm = confusion_matrix(y_test, y_pred)

  for i in range(len(cm)):
      total = 0
      for m in cm[i]:
        total+=m
      parts.append(float((1.0-(cm[i][i]/total)) * 100))

  if len(cm) == 3:
    return parts[0], parts[2], parts[1]
  elif len(cm) == 2:
    return parts[0], parts[1]

In [4]:
# Random Forest 1 - Imbalanced model predicting all 3 classes

rf_1 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight=None, min_samples_leaf=2, min_samples_split=5, max_features='sqrt', max_depth=10)

# getting the data ready to use
X, y = scale_data(preprocess_dataset1(get_data()))
X_new, y_new = scale_data(preprocess_dataset2(new_get_data()))

# splitting the train and tests portions
rf1_X_train, rf1_X_test, rf1_y_train, rf1_y_test = train_test_split(X, y, random_state=42, test_size=0.25)
rf1_new_X_train, rf1_new_X_test, rf1_new_y_train, rf1_new_y_test = train_test_split(X_new, y_new, random_state=42, test_size=0.25)

# training the model
rf_1.fit(rf1_X_train, rf1_y_train)

rf_y_pred1_1 = rf_1.predict(rf1_X_test)
rf_y_pred1_2 = rf_1.predict(rf1_new_X_test)

print_metrics(rf1_y_test, rf_y_pred1_1, 'dataset 2015-2018')
print_metrics(rf1_new_y_test, rf_y_pred1_2, 'dataset 2023-2025')

/tmp/ipykernel_6883/3753761261.py:30: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/kaggle/input/boston-crime-data/crime.csv', encoding='latin-1')
/tmp/ipykernel_6883/787746400.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Sunday', 1)


UCR_PART
Part Three    161903
Part Two       99765
Part One       63011
Name: count, dtype: int64
UCR_PART
Part Three    133003
Part Two       96016
Part One       43644
Name: count, dtype: int64




# Results for dataset 2015-2018

       Accuracy: 49.71%
       F1 macro: 23.45%
Precision macro: 40.41%
   Recall macro: 33.62%



  Part One Error: 99.57%
  Part Two Error: 98.34%
Part Three Error: 1.24%

 Confusion Matrix
            Part One     Part Two     Part Three
  Part One      68         243           15478
  Part Two      57         415           24542
Part Three      83         416           39868




# Results for dataset 2023-2025

       Accuracy: 48.71%
       F1 macro: 22.56%
Precision macro: 38.91%
   Recall macro: 33.44%



  Part One Error: 99.87%
  Part Two Error: 98.97%
Part Three Error: 0.83%

 Confusion Matrix
            Part One     Part Two     Part Three
  Part One      14         107           10743
  Part Two      17         247           23819
Part Three   

In [5]:
# Random Forest 2 - Balanced model predicting all 3 classes

rf_2 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_leaf=2, min_samples_split=5, max_features='sqrt', max_depth=10)

# getting the data ready to use
X, y = scale_data(preprocess_dataset1(get_data()))
X_new, y_new = scale_data(preprocess_dataset2(new_get_data()))

# splitting the train and tests portions
rf2_X_train, rf2_X_test, rf2_y_train, rf2_y_test = train_test_split(X, y, random_state=42, test_size=0.25)
rf2_new_X_train, rf2_new_X_test, rf2_new_y_train, rf2_new_y_test = train_test_split(X_new, y_new, random_state=42, test_size=0.25)

# training the model
rf_2.fit(rf2_X_train, rf2_y_train)

rf_y_pred2_1 = rf_2.predict(rf2_X_test)
rf_y_pred2_2 = rf_2.predict(rf2_new_X_test)

print_metrics(rf2_y_test, rf_y_pred2_1, 'dataset 2015-2018')
print_metrics(rf2_new_y_test, rf_y_pred2_2, 'dataset 2023-2025')


/tmp/ipykernel_6883/3753761261.py:30: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/kaggle/input/boston-crime-data/crime.csv', encoding='latin-1')
/tmp/ipykernel_6883/787746400.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Sunday', 1)


UCR_PART
Part Three    161903
Part Two       99765
Part One       63011
Name: count, dtype: int64
UCR_PART
Part Three    133003
Part Two       96016
Part One       43644
Name: count, dtype: int64




# Results for dataset 2015-2018

       Accuracy: 40.47%
       F1 macro: 38.81%
Precision macro: 39.20%
   Recall macro: 39.85%



  Part One Error: 63.47%
  Part Two Error: 58.08%
Part Three Error: 58.89%

 Confusion Matrix
            Part One     Part Two     Part Three
  Part One      5767         5106           4916
  Part Two      6237         10486           8291
Part Three      9622         14149           16596




# Results for dataset 2023-2025

       Accuracy: 38.44%
       F1 macro: 36.44%
Precision macro: 36.91%
   Recall macro: 37.89%



  Part One Error: 63.32%
  Part Two Error: 63.33%
Part Three Error: 59.69%

 Confusion Matrix
            Part One     Part Two     Part Three
  Part One      3985         3409           3470
  Part Two      6673         8831           857

In [6]:
# Random Forest 3 - Balanced model predicting only 2 classes

rf_3 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_leaf=2, min_samples_split=5, max_features='sqrt', max_depth=10)

# getting the data ready to use
X, y = scale_data(preprocess_dataset1(get_data(), all_classes=False))
X_new, y_new = scale_data(preprocess_dataset2(new_get_data(), all_classes=False))

# splitting the train and tests portions
rf3_X_train, rf3_X_test, rf3_y_train, rf3_y_test = train_test_split(X, y, random_state=42, test_size=0.25)
rf3_new_X_train, rf3_new_X_test, rf3_new_y_train, rf3_new_y_test = train_test_split(X_new, y_new, random_state=42, test_size=0.25)

# training the model
rf_3.fit(rf3_X_train, rf3_y_train)

rf_y_pred3_1 = rf_3.predict(rf3_X_test)
rf_y_pred3_2 = rf_3.predict(rf3_new_X_test)

print_metrics(rf3_y_test, rf_y_pred3_1, 'dataset 2015-2018')
print_metrics(rf3_new_y_test, rf_y_pred3_2, 'dataset 2023-2025')



/tmp/ipykernel_6883/3753761261.py:30: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/kaggle/input/boston-crime-data/crime.csv', encoding='latin-1')
/tmp/ipykernel_6883/787746400.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Sunday', 1)


UCR_PART
Part Two    99765
Part One    63011
Name: count, dtype: int64
UCR_PART
Part Two    96016
Part One    43644
Name: count, dtype: int64




# Results for dataset 2015-2018

       Accuracy: 58.17%
       F1 macro: 56.40%
Precision macro: 56.37%
   Recall macro: 56.50%



Part One Error: 50.84%
Part Two Error: 36.15%

 Confusion Matrix
            Part One     Part Two
Part One      7734         7999
Part Two      9024         15937




# Results for dataset 2023-2025

       Accuracy: 56.34%
       F1 macro: 53.06%
Precision macro: 53.52%
   Recall macro: 53.99%



Part One Error: 52.29%
Part Two Error: 39.72%

 Confusion Matrix
            Part One     Part Two
Part One      5224         5725
Part Two      9520         14446


In [9]:
# Random Forest 4 - Balanced model predicting all 3 with UNDERSAMPLING

rf_4 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight=None, min_samples_leaf=2, min_samples_split=5, max_features='sqrt', max_depth=10)

# getting the data ready to use
X, y = scale_data(preprocess_dataset1(get_data()), option='undersample')
X_new, y_new = scale_data(preprocess_dataset2(new_get_data()), option='undersample')

# splitting the train and tests portions
rf4_X_train, rf4_X_test, rf4_y_train, rf4_y_test = train_test_split(X, y, random_state=42, test_size=0.25)
rf4_new_X_train, rf4_new_X_test, rf4_new_y_train, rf4_new_y_test = train_test_split(X_new, y_new, random_state=42, test_size=0.25)

# training the model
rf_4.fit(rf4_X_train,rf4_y_train)

rf_y_pred4_1 = rf_4.predict(rf4_X_test)
rf_y_pred4_2 = rf_4.predict(rf4_new_X_test)

print_metrics(rf4_y_test, rf_y_pred4_1, 'dataset 2015-2018')
print_metrics(rf4_new_y_test, rf_y_pred4_2, 'dataset 2023-2025')

/tmp/ipykernel_6883/3753761261.py:30: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/kaggle/input/boston-crime-data/crime.csv', encoding='latin-1')
/tmp/ipykernel_6883/787746400.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Sunday', 1)


UCR_PART
Part One      63011
Part Two      63011
Part Three    63011
Name: count, dtype: int64
UCR_PART
Part One      43644
Part Two      43644
Part Three    43644
Name: count, dtype: int64




# Results for dataset 2015-2018

       Accuracy: 39.50%
       F1 macro: 39.49%
Precision macro: 39.68%
   Recall macro: 39.52%



  Part One Error: 62.33%
  Part Two Error: 56.86%
Part Three Error: 62.27%

 Confusion Matrix
            Part One     Part Two     Part Three
  Part One      5924         5285           4515
  Part Two      4065         6721           4792
Part Three      4056         5880           6021




# Results for dataset 2023-2025

       Accuracy: 37.31%
       F1 macro: 37.34%
Precision macro: 37.48%
   Recall macro: 37.31%



  Part One Error: 62.56%
  Part Two Error: 61.76%
Part Three Error: 63.76%

 Confusion Matrix
            Part One     Part Two     Part Three
  Part One      4126         3752           3141
  Part Two      3293         4133           3381
Part Th

In [10]:
# Random Forest 5 - Balanced model predicting all 3 with SMOTE (OVERSAMPLING)

rf_5 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight=None, min_samples_leaf=2, min_samples_split=5, max_features='sqrt', max_depth=10)

# getting the data ready to use
X, y = scale_data(preprocess_dataset1(get_data()), option='oversample')
X_new, y_new = scale_data(preprocess_dataset2(new_get_data()), option='oversample')

# splitting the train and tests portions
rf5_X_train, rf5_X_test, rf5_y_train, rf5_y_test = train_test_split(X, y, random_state=42, test_size=0.25)
rf5_new_X_train, rf5_new_X_test, rf5_new_y_train, rf5_new_y_test = train_test_split(X_new, y_new, random_state=42, test_size=0.25)

# training the model
rf_5.fit(rf5_X_train, rf5_y_train)

rf_y_pred5_1 = rf_5.predict(rf5_X_test)
rf_y_pred5_2 = rf_5.predict(rf5_new_X_test)

print_metrics(rf5_y_test, rf_y_pred5_1, 'dataset 2015-2018')
print_metrics(rf5_new_y_test, rf_y_pred5_2, 'dataset 2023-2025')

/tmp/ipykernel_6883/3753761261.py:30: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/kaggle/input/boston-crime-data/crime.csv', encoding='latin-1')
/tmp/ipykernel_6883/787746400.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['DAY_OF_WEEK'] = df['DAY_OF_WEEK'].replace('Sunday', 1)


UCR_PART
Part Two      161903
Part Three    161903
Part One      161903
Name: count, dtype: int64
UCR_PART
Part One      133003
Part Two      133003
Part Three    133003
Name: count, dtype: int64




# Results for dataset 2015-2018

       Accuracy: 41.11%
       F1 macro: 41.14%
Precision macro: 41.25%
   Recall macro: 41.11%



  Part One Error: 59.31%
  Part Two Error: 57.47%
Part Three Error: 59.89%

 Confusion Matrix
            Part One     Part Two     Part Three
  Part One      16328         12422           11373
  Part Two      10642         17277           12704
Part Three      10035         14330           16317




# Results for dataset 2023-2025

       Accuracy: 37.80%
       F1 macro: 37.77%
Precision macro: 37.93%
   Recall macro: 37.80%



  Part One Error: 64.31%
  Part Two Error: 64.74%
Part Three Error: 57.55%

 Confusion Matrix
            Part One     Part Two     Part Three
  Part One      11830         10585           10733
  Part Two      9149         11757    

In [11]:
# dataframe with all results for the 2015-2018 dataset

p1_1, p2_1, p3_1 = separate_confusions(rf1_y_test, rf_y_pred1_1)
p1_2, p2_2, p3_2 = separate_confusions(rf2_y_test, rf_y_pred2_1)
p1_3, p2_3 = separate_confusions(rf3_y_test, rf_y_pred3_1)
p1_4, p2_4, p3_4 = separate_confusions(rf4_y_test, rf_y_pred4_1)
p1_5, p2_5, p3_5 = separate_confusions(rf5_y_test, rf_y_pred5_1)

results = pd.DataFrame({
    'Model': ['Random Forest 1 (IMBALANCED)', 'Random Forest 2 (BALANCED)', 'Random Forest 3 (2 CLASSES, BALANCED)', 'Random Forest 4 (UNDERSAMPLING)', 'Random Forest 5 (OVERSAMPLING)'],
    'Accuracy': [f"{(accuracy_score(rf1_y_test, rf_y_pred1_1)*100):.2f}%", f"{(accuracy_score(rf2_y_test, rf_y_pred2_1)*100):.2f}%", f"{(accuracy_score(rf3_y_test, rf_y_pred3_1)*100):.2f}%", f"{(accuracy_score(rf4_y_test, rf_y_pred4_1)*100):.2f}%", f"{(accuracy_score(rf5_y_test, rf_y_pred5_1)*100):.2f}%"],
    'F1-macro': [f"{(f1_score(rf1_y_test, rf_y_pred1_1, average='macro')*100):.2f}%", f"{(f1_score(rf2_y_test, rf_y_pred2_1, average='macro')*100):.2f}%", f"{(f1_score(rf3_y_test, rf_y_pred3_1, average='macro')*100):.2f}%", f"{(f1_score(rf4_y_test, rf_y_pred4_1, average='macro')*100):.2f}%", f"{(f1_score(rf5_y_test, rf_y_pred5_1, average='macro')*100):.2f}%"],
    'Precision-macro': [f"{(precision_score(rf1_y_test, rf_y_pred1_1, average='macro')*100):.2f}%", f"{(precision_score(rf2_y_test, rf_y_pred2_1, average='macro')*100):.2f}%", f"{(precision_score(rf3_y_test, rf_y_pred3_1, average='macro')*100):.2f}%", f"{(precision_score(rf4_y_test, rf_y_pred4_1, average='macro')*100):.2f}%", f"{(precision_score(rf5_y_test, rf_y_pred5_1, average='macro')*100):.2f}%"],
    'Recall-macro': [f"{(recall_score(rf1_y_test, rf_y_pred1_1, average='macro')*100):.2f}%", f"{(recall_score(rf2_y_test, rf_y_pred2_1, average='macro')*100):.2f}%", f"{(recall_score(rf3_y_test, rf_y_pred3_1, average='macro')*100):.2f}%", f"{(recall_score(rf4_y_test, rf_y_pred4_1, average='macro')*100):.2f}%", f"{(recall_score(rf5_y_test, rf_y_pred5_1, average='macro')*100):.2f}%"],
    'Part One Error': [f"{p1_1:.2f}%", f"{p1_2:.2f}%", f"{p1_3:.2f}%", f"{p1_4:.2f}%", f"{p1_5:.2f}%"],
    'Part Two Error': [f"{p2_1:.2f}%", f"{p2_2:.2f}%", f"{p2_3:.2f}%", f"{p2_4:.2f}%", f"{p2_5:.2f}%"],
    'Part Three Error': [f"{p3_1:.2f}%", f"{p3_2:.2f}%", f"---", f"{p3_4:.2f}%", f"{p3_5:.2f}%"]
})

results

,Model,Accuracy,F1-macro,Precision-macro,Recall-macro,Part One Error,Part Two Error,Part Three Error
0,Random Forest 1 (IMBALANCED),49.71%,23.45%,40.41%,33.62%,99.57%,98.34%,1.24%
1,Random Forest 2 (BALANCED),40.47%,38.81%,39.20%,39.85%,63.47%,58.08%,58.89%
2,"Random Forest 3 (2 CLASSES, BALANCED)",58.17%,56.40%,56.37%,56.50%,50.84%,36.15%,---
3,Random Forest 4 (UNDERSAMPLING),39.50%,39.49%,39.68%,39.52%,62.33%,56.86%,62.27%
4,Random Forest 5 (OVERSAMPLING),41.11%,41.14%,41.25%,41.11%,59.31%,57.47%,59.89%


In [12]:
# dataframe with all results for the 2023-2025 dataset

p1_1, p2_1, p3_1 = separate_confusions(rf1_new_y_test, rf_y_pred1_2)
p1_2, p2_2, p3_2 = separate_confusions(rf2_new_y_test, rf_y_pred2_2)
p1_3, p2_3 = separate_confusions(rf3_new_y_test, rf_y_pred3_2)
p1_4, p2_4, p3_4 = separate_confusions(rf4_new_y_test, rf_y_pred4_2)
p1_5, p2_5, p3_5 = separate_confusions(rf5_new_y_test, rf_y_pred5_2)

results2 = pd.DataFrame({
    'Model': ['Random Forest 1 (IMBALANCED)', 'Random Forest 2 (BALANCED)', 'Random Forest 3 (2 CLASSES, BALANCED)', 'Random Forest 4 (UNDERSAMPLING)', 'Random Forest 5 (OVERSAMPLING)'],
    'Accuracy': [f"{(accuracy_score(rf1_new_y_test, rf_y_pred1_2)*100):.2f}%", f"{(accuracy_score(rf2_new_y_test, rf_y_pred2_2)*100):.2f}%", f"{(accuracy_score(rf3_new_y_test, rf_y_pred3_2)*100):.2f}%", f"{(accuracy_score(rf4_new_y_test, rf_y_pred4_2)*100):.2f}%", f"{(accuracy_score(rf5_new_y_test, rf_y_pred5_2)*100):.2f}%"],
    'F1-macro': [f"{(f1_score(rf1_new_y_test, rf_y_pred1_2, average='macro')*100):.2f}%", f"{(f1_score(rf2_new_y_test, rf_y_pred2_2, average='macro')*100):.2f}%", f"{(f1_score(rf3_new_y_test, rf_y_pred3_2, average='macro')*100):.2f}%", f"{(f1_score(rf4_new_y_test, rf_y_pred4_2, average='macro')*100):.2f}%", f"{(f1_score(rf5_new_y_test, rf_y_pred5_2, average='macro')*100):.2f}%"],
    'Precision-macro': [f"{(precision_score(rf1_new_y_test, rf_y_pred1_2, average='macro')*100):.2f}%", f"{(precision_score(rf2_new_y_test, rf_y_pred2_2, average='macro')*100):.2f}%", f"{(precision_score(rf3_new_y_test, rf_y_pred3_2, average='macro')*100):.2f}%", f"{(precision_score(rf4_new_y_test, rf_y_pred4_2, average='macro')*100):.2f}%", f"{(precision_score(rf5_new_y_test, rf_y_pred5_2, average='macro')*100):.2f}%"],
    'Recall-macro': [f"{(recall_score(rf1_new_y_test, rf_y_pred1_2, average='macro')*100):.2f}%", f"{(recall_score(rf2_new_y_test, rf_y_pred2_2, average='macro')*100):.2f}%", f"{(recall_score(rf3_new_y_test, rf_y_pred3_2, average='macro')*100):.2f}%", f"{(recall_score(rf4_new_y_test, rf_y_pred4_2, average='macro')*100):.2f}%", f"{(recall_score(rf5_new_y_test, rf_y_pred5_2, average='macro')*100):.2f}%"],
    'Part One Error': [f"{p1_1:.2f}%", f"{p1_2:.2f}%", f"{p1_3:.2f}%", f"{p1_4:.2f}%", f"{p1_5:.2f}%"],
    'Part Two Error': [f"{p2_1:.2f}%", f"{p2_2:.2f}%", f"{p2_3:.2f}%", f"{p2_4:.2f}%", f"{p2_5:.2f}%"],
    'Part Three Error': [f"{p3_1:.2f}%", f"{p3_2:.2f}%", f"---", f"{p3_4:.2f}%", f"{p3_5:.2f}%"]
})

results2

,Model,Accuracy,F1-macro,Precision-macro,Recall-macro,Part One Error,Part Two Error,Part Three Error
0,Random Forest 1 (IMBALANCED),48.71%,22.56%,38.91%,33.44%,99.87%,98.97%,0.83%
1,Random Forest 2 (BALANCED),38.44%,36.44%,36.91%,37.89%,63.32%,63.33%,59.69%
2,"Random Forest 3 (2 CLASSES, BALANCED)",56.34%,53.06%,53.52%,53.99%,52.29%,39.72%,---
3,Random Forest 4 (UNDERSAMPLING),37.31%,37.34%,37.48%,37.31%,62.56%,61.76%,63.76%
4,Random Forest 5 (OVERSAMPLING),37.80%,37.77%,37.93%,37.80%,64.31%,64.74%,57.55%
